In [34]:
%load_ext byname

The byname extension is already loaded. To reload it, use:
  %reload_ext byname


In [35]:
from notebook_cache import NotebookCache

# results and prints live on disk, so they survive kernel restarts; one folder per notebook
cache = NotebookCache("../cache/act_byname")

# ACT, built forward (byname + cache)

Redo of `act.ipynb`: same data, split and eval seeds. Expensive steps are `@cache`d, so a kernel restart costs nothing.

### step 0 — the data, as in BC


In [36]:
from pathlib import Path

import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from numpy.typing import NDArray

np.set_printoptions(precision=3, suppress=True)
REPO = Path.cwd().parent


@cache
def load_table():
    # straight from the table, no video decode
    ds = LeRobotDataset("dream_robot/pick_place_cube", root=REPO / "data/robosuite/pick_place_cube")
    return (
        state=np.asarray(ds.hf_dataset["observation.state"], dtype=np.float32),  # [N, 8]
        action=np.asarray(ds.hf_dataset["action"], dtype=np.float32),  # [N, 8]
        episode=np.asarray(
            ds.hf_dataset["episode_index"], dtype=np.int64
        ),  # [N], which demo each frame is from
    )


(state=, action=, episode=) = load_table()
val_eps = np.random.default_rng(0).permutation(25)[:5]  # same 5 held-out demos as bc_2 and act
is_val = np.isin(episode, val_eps)  # [N]
print(f"{state.shape=}  {action.shape=}  {val_eps=}  train {(~is_val).sum()}  val {is_val.sum()}")


state.shape=(5288, 8)  action.shape=(5288, 8)  val_eps=array([19,  4, 10, 11, 24])  train 4218  val 1070


### step 1 — the target is a chunk: the next K actions, not one


In [37]:
def make_chunks(action: NDArray[np.float32], episode: NDArray[np.int64], K: int):
    # demo e's last row is where its run of rows ends, since demos are stored in order
    assert (np.diff(episode) >= 0).all(), "demos not stored in order"
    last = np.searchsorted(episode, episode, side="right") - 1  # [N]
    idx = np.arange(len(action))[:, None] + np.arange(K)  # [N, K], rows t .. t+K-1
    pad = idx > last[:, None]  # [N, K], True where the chunk runs past its demo's end
    chunks = action[
        np.minimum(idx, last[:, None])
    ]  # [N, K, 8], past the end: repeat the final action
    return (chunks=, pad=)


K = 50  # chunk length: 50 steps = 1.7 s at 30 Hz; a demo is ~210 steps
(chunks=, pad=) = make_chunks(action, episode, K)
print(f"{chunks.shape=}  {pad.shape=}")
print(f"frames whose chunk is partly padding: {pad.any(1).sum()} of {len(action)}")


chunks.shape=(5288, 50, 8)  pad.shape=(5288, 50)
frames whose chunk is partly padding: 1225 of 5288
